# UnderCover: deciding claims with an LLM

Week 1 Day 2. UnderCover Travel Insurance has a queue of twenty-five claims, and a claims handler has already decided each one. We put every claim to a line-up of models through OpenRouter, with the customer's policy record and the policy wording. Each model replies with a structured decision: approve, decline or refer, with a rationale and a payout. We score the answers against the handlers' decisions and look at what each model cost.

Then we check the score on twenty-five held-out claims, try two optional extras, and finish by giving the same job to a LangChain agent that looks up the policy and records its decision with tools.

In [1]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

from sample_data import DATA, get_sample_data, load

## Connect to OpenRouter and Ollama

The OpenRouter key comes from the `.env` file in this folder. One entry in the line-up runs on this machine through Ollama, which serves the same OpenAI API locally. It needs `ollama pull gpt-oss`, about 13 GB. If you don't run Ollama, skip the cell that follows the clients and leave that entry out of the line-up.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama", timeout=600, max_retries=0)

OpenRouter API key found


Ollama loads a model with room for 4,096 tokens unless told otherwise, and it stops a reply that runs past that. Our prompt is about 3,500 tokens before the model starts to reason, so we make a copy of `gpt-oss` with a 32,768 token context. The copy shares the model's weights and takes no extra disk space.

In [ ]:
copy = {"model": "gpt-oss-32k", "from": "gpt-oss", "parameters": {"num_ctx": 32768}, "stream": False}
response = requests.post("http://localhost:11434/api/create", json=copy)
print(response.json())

## Load the seed data

`get_sample_data` returns a list of claims and the handlers' decisions on them, keyed by claim id. The claims, the policies and the policy wording are what the model sees. The decisions are the labels. They stay out of every prompt and are only used for scoring.

The default is the queue of twenty-five claims, which is the set we work against. `tiny=True` keeps five of them, which fits inside a free model's daily allowance.

In [4]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text()

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [5]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

The handler's decision on the same claim. The tags at the end describe the claim for our analysis later.

In [6]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

## The structured output

A Pydantic class describes the reply we want. The rationale comes first so that the model reasons before it commits to a decision.

In [7]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="Two or three sentences that explain the decision and name the section of the policy wording it rests on")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt holds the task and the whole policy wording, which is short enough to fit. The user prompt holds the policy record and the claim as JSON.

In [8]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the documents it asks for are attached. Work out the payout the way section 10 describes.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Give your rationale first, then the decision, then the payout.

{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

## One call with a structured output

`parse` sends the Pydantic class as a JSON schema and hands back an instance of it. OpenRouter reports what each call cost in `usage.cost`, in dollars.

OpenRouter spreads an open model like gpt-oss-20b across a dozen providers, and they differ in precision and in how well they handle structured outputs. In testing, one provider returned an empty reply, one filled in the decision and left the rationale blank, and one rejected the request, although all three advertise structured outputs. So we name the providers we have tested. OpenRouter tries them in order and moves to the next when one is busy, which keeps the results repeatable from run to run.

These models all reason before they answer, and their default levels differ, so we always set the level. `low` keeps a run quick and cheap, and we raise the level where a model needs it.

In [9]:
PROVIDERS = {
    "openai/gpt-oss-20b": ["deepinfra", "coreweave", "parasail"],
    "qwen/qwen3.8-27b": ["deepinfra", "akashml", "chutes"],
}


def provider_settings(model):
    """Route only to providers that support structured outputs, and keep to the tested providers where we have named them."""
    settings = {"require_parameters": True}
    if model in PROVIDERS:
        settings["order"] = PROVIDERS[model]
        settings["allow_fallbacks"] = False
    return settings


def decide(claim, model, effort="low", local=False):
    """Ask one model to decide one claim. local=True sends the call to Ollama on this machine, which costs nothing."""
    client = ollama if local else openrouter
    routing = {} if local else {"provider": provider_settings(model)}
    start = time.time()
    response = client.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body=routing,
    )
    answer = response.choices[0].message.parsed
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "provider": "ollama" if local else response.provider,
        "cost": 0.0 if local else response.usage.cost,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "seconds": round(time.time() - start, 1),
    }

In [10]:
first = decide(claims[0], "openai/gpt-5.6-luna")
first

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'The cancellation was caused by a crew problem and delayed departure from June 18 at 11:40 to June 19 at 06:55, exceeding the Premium plan’s six-hour waiting period under Section 3 (Trip Delay). The attached carrier confirmation and receipts support the $420.00 of reasonable accommodation, meals, and local transport expenses, payable under Section 10 up to the $800 maximum.',
 'provider': 'OpenAI',
 'cost': 0.0009178,
 'input_tokens': 2779,
 'output_tokens': 186,
 'seconds': 3.2}

## The line-up

Five entries, all on the same prompt. GPT-5.6 Luna and Qwen 3.8 27B run at the `low` reasoning level. gpt-oss-20b is a much smaller model and does better with more room to reason, so it runs at `medium`, once hosted and once on this machine through Ollama, which is the same open-weight model at no cost. Luna runs again at `medium`, to see what more reasoning buys.

Every claim goes to every entry, one call per claim. OpenRouter takes five calls at a time and Ollama takes one, so the local entry is the slow one, at around twenty seconds a claim on an M1 Max.

In [11]:
LINEUP = {
    "Luna low": {"model": "openai/gpt-5.6-luna", "effort": "low"},
    "Qwen low": {"model": "qwen/qwen3.8-27b", "effort": "low"},
    "gpt-oss-20b medium": {"model": "openai/gpt-oss-20b", "effort": "medium"},
    "gpt-oss-20b medium, local": {"model": "gpt-oss-32k", "effort": "medium", "local": True},
    "Luna medium": {"model": "openai/gpt-5.6-luna", "effort": "medium"},
}


def run(name, claims):
    """Decide every claim with one entry of the line-up."""
    settings = LINEUP[name]
    workers = 1 if settings.get("local") else 5
    start = time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        rows = list(pool.map(lambda claim: decide(claim, **settings), claims))
    print(f"{name}: {len(claims)} claims in {time.time() - start:.0f} seconds")
    return [{"model": name, **row} for row in rows]

In [12]:
results = []
for name in LINEUP:
    results += run(name, claims)

Luna low: 25 claims in 22 seconds


Qwen low: 25 claims in 153 seconds


gpt-oss-20b medium: 25 claims in 36 seconds


gpt-oss-20b medium, local: 25 claims in 559 seconds


Luna medium: 25 claims in 25 seconds


## Score against the handlers

The decision is right when it matches the handler's. The payout is right when the decision is right and the amount is within a dollar of the handler's.

In [13]:
def score(results, labels):
    """Mark each answer against the handler's decision on the same claim."""
    scored = pd.DataFrame(results)
    scored["label"] = scored["claim_id"].map(lambda claim_id: labels[claim_id]["decision"])
    scored["label_payout"] = scored["claim_id"].map(lambda claim_id: labels[claim_id]["payout"])
    scored["difficulty"] = scored["claim_id"].map(lambda claim_id: labels[claim_id]["difficulty"])
    scored["decision_right"] = scored["decision"] == scored["label"]
    scored["payout_right"] = scored["decision_right"] & ((scored["payout"] - scored["label_payout"]).abs() <= 1)
    return scored


def summarize(scored):
    """One row per entry: what it got right, what it cost and how long a claim took."""
    return scored.groupby("model", sort=False).agg(
        decisions_right=("decision_right", "sum"),
        payouts_right=("payout_right", "sum"),
        claims=("claim_id", "count"),
        cost=("cost", "sum"),
        output_tokens=("output_tokens", "sum"),
        mean_seconds=("seconds", "mean"),
    )


scored = score(results, labels)
summarize(scored)

,decisions_right,payouts_right,claims,cost,output_tokens,mean_seconds
model,,,,,,
Luna low,18,16,25,0.010831,4419,3.296
Qwen low,21,20,25,0.069774,33485,27.808
gpt-oss-20b medium,17,13,25,0.004893,22073,6.448
"gpt-oss-20b medium, local",16,12,25,0.000000,5151,22.344
Luna medium,19,17,25,0.012705,5449,3.528


The claims that were fully right, by the kind of claim. The policy wording settles the routine claims and the hard ones. The manual claims turn on the claims manual or the customer's history, and this prompt holds neither, so expect nearly all of them to be wrong at this stage. The queue has 15 routine, 5 hard and 5 manual claims.

In [14]:
def by_difficulty(scored):
    """Count the claims with the decision and the payout right, by the kind of claim."""
    table = scored.pivot_table(index="model", columns="difficulty", values="payout_right", aggfunc="sum", sort=False)
    return table[["routine", "hard", "manual"]]


by_difficulty(scored)

difficulty,routine,hard,manual
model,,,
Luna low,14,2,0
Qwen low,15,4,1
gpt-oss-20b medium,13,0,0
"gpt-oss-20b medium, local",12,0,0
Luna medium,15,2,0


Which provider served each call. When the first provider on our list is busy, OpenRouter moves to the next.

In [15]:
scored.groupby(["model", "provider"], sort=False).size()

model                      provider 
Luna low                   OpenAI       25
Qwen low                   DeepInfra    25
gpt-oss-20b medium         CoreWeave    25
gpt-oss-20b medium, local  ollama       25
Luna medium                OpenAI       25
dtype: int64

## Side by side

One row per claim, with the handler's decision and each entry's answer.

In [16]:
scored["answer"] = scored["decision"] + " $" + scored["payout"].round(2).astype(str)
side_by_side = scored.pivot(index="claim_id", columns="model", values="answer")[scored["model"].unique()]
side_by_side.insert(0, "handler", [f"{labels[claim_id]['decision']} ${labels[claim_id]['payout']}" for claim_id in side_by_side.index])
summaries = {claim["claim_id"]: claim["summary"] for claim in claims}
side_by_side.insert(0, "summary", [summaries[claim_id] for claim_id in side_by_side.index])
pd.set_option("display.max_colwidth", 60)
side_by_side

model,summary,handler,Luna low,Qwen low,gpt-oss-20b medium,"gpt-oss-20b medium, local",Luna medium
claim_id,,,,,,,
CLM-1001,"Flight home canceled by the airline, rebooked the next m...",approve $420.0,approve $420.0,approve $420.0,approve $420.0,approve $420.0,approve $420.0
CLM-1002,Suitcase lost on a connecting flight,approve $580.0,refer $0.0,approve $580.0,refer $0.0,refer $0.0,approve $580.0
CLM-1003,"Hospital visit abroad after a fall, fractured wrist",approve $3750.0,approve $3750.0,approve $3750.0,approve $3750.0,approve $3750.0,approve $3750.0
CLM-1004,Checked bag arrived 31 hours late,approve $186.0,approve $186.0,approve $186.0,approve $186.0,approve $186.0,approve $186.0
CLM-1005,Passport eaten by a giraffe at breakfast,approve $243.0,approve $243.0,approve $243.0,approve $243.0,approve $243.0,approve $243.0
CLM-1006,Hit by a falling coconut while reading under a palm tree,approve $425.0,approve $425.0,approve $425.0,approve $425.0,approve $425.0,approve $425.0
CLM-1007,Phone dropped in the hotel shower,approve $50.0,approve $50.0,approve $50.0,approve $50.0,approve $50.0,approve $50.0
CLM-1008,Trip to Seattle canceled after a tornado carried off her...,approve $2240.0,approve $2240.0,approve $2240.0,approve $2440.0,approve $2440.0,approve $2240.0
CLM-1009,Camera snatched on the Barcelona metro,approve $930.0,approve $930.0,approve $930.0,approve $930.0,approve $930.0,approve $930.0


## Where a model got it wrong

Each miss with the model's rationale, beside the handler's reason and what the claim is in the set to test. We look at Luna at `low`. Change the name to read another entry's misses.

In [17]:
def show_misses(scored, labels, name):
    """Show every claim that one entry got wrong, with its rationale and the handler's reason."""
    misses = scored[(scored["model"] == name) & ~scored["payout_right"]].sort_values("claim_id")
    for miss in misses.to_dict("records"):
        label = labels[miss["claim_id"]]
        display(Markdown(f"""**{miss['claim_id']}, {label['difficulty']}**: {miss['decision']} ${miss['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {miss['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(scored, labels, "Luna low")

**CLM-1002, routine**: refer $0.00, where the handler said approve $580.00

- Model: The lost suitcase and contents are covered under Section 5, and the carrier loss report is attached. However, the claim is missing the required repair estimate or non-repairable statement only for accidental damage, which is not relevant here; the stated missing receipts for the shirts and toiletry kit are not documents expressly required by Section 5. Eligible items are limited by the Premium per-item limit of $300: suitcase $280, wool suit $300, dress shoes $160, running shoes $110, shirts $120, and toiletry kit $60, totaling $1,030. After deducting the carrier’s $400 payment and the $50 Premium baggage deductible under Section 10, the payable amount is $580, subject to confirmation that the undocumented items were part of the lost checked bag; because the submitted contents and receipts do not fully document those items, this claim should be referred rather than paid immediately.
- Handler: The loss was reported to the carrier and the carrier has declared the bag lost. The suit is over the $300 per-item limit on Premium, so it counts at $300. The carrier's $400 comes off, then the $50 deductible.
- Tests: Payout arithmetic: the per-item limit, the carrier's payment and the deductible in order


**CLM-1011, hard**: approve $1,110.00, where the handler said approve $1,150.00

- Model: The lost checked baggage is covered under Section 5, and the carrier report and settlement letter document the loss and the $350 carrier payment. Applying Section 10, the eligible amounts are capped at the Premium per-item limit of $300, with the tuxedo and camera with lens each treated as one set; after deducting the carrier payment and the $50 Premium deductible, the payout is $1,110.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The theft occurred during the covered trip and was reported to the local police within 24 hours, with the required police report attached, so it is covered under Section 5, Baggage and Personal Effects. Under Section 10, the leather jacket is limited to the Premium plan’s $300 per-item limit and the daypack is eligible at $80; after the $50 deductible, the payable amount is $330.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The 9-hour carrier delay was caused by a crew problem, which is covered under Section 3 (Trip Delay), and it exceeds the Premium plan’s 6-hour waiting period. The attached carrier confirmation and receipts support reasonable accommodation, meal, and local transport expenses totaling $210; under Section 10, no deductible applies and the amount is within the $800 maximum.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The loss occurred during the covered trip and baggage cover may apply under Section 5, but the claim describes theft and Section 5 requires the theft to be reported to local police within 24 hours with a copy of the police report. The attached tour-guide letter does not meet that requirement, so the claim must wait for the required document or further review under Section 11.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft occurred during the covered trip, and the police report confirms it was reported on July 6, within 24 hours as required by Section 5 (Baggage and Personal Effects). The items were not left unattended, and Electronics Plus raises the per-item limit for the three electronics; under Section 10, the $1,103 eligible claim is reduced by the Premium plan’s $50 deductible.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: The 30-hour delay is potentially covered under Section 3 because the carrier confirms a severe-weather delay, exceeding the First plan’s 3-hour waiting period. However, Section 3 requires receipts for reasonable extra costs, and no receipts were provided for the $140 meal expense; the claim therefore requires further documentation before payment can be determined.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: The nine-hour delay exceeds the First plan’s three-hour waiting period and was caused by a mechanical problem, which is covered under Section 3 (Trip Delay). The carrier delay certificate and meal receipts are attached as required; under Section 10, the eligible expenses total $61, below the $1,500 maximum, with no deductible applying.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The phone was stolen during the covered trip and was not left unattended, since it was in Isabel’s hand. Theft was reported to local police the same day, and the required police report and purchase receipt are attached, satisfying Section 5. Under Section 10, the eligible amount is limited to the Premium plan’s $300 per-item limit, less the $50 deductible under Section 1, with no carrier payment to deduct.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


## What it cost

OpenRouter's reported cost for every call so far, in dollars.

In [18]:
total = scored["cost"].sum() + first["cost"]
print(f"Total for {len(scored) + 1} calls: ${total:.4f}")
scored.groupby("model", sort=False)["cost"].agg(["sum", "mean", "max"])

Total for 126 calls: $0.0991


,sum,mean,max
model,,,
Luna low,0.010831,0.000433,0.000889
Qwen low,0.069774,0.002791,0.005822
gpt-oss-20b medium,0.004893,0.000196,0.000365
"gpt-oss-20b medium, local",0.000000,0.000000,0.000000
Luna medium,0.012705,0.000508,0.000880


## The held-out claims

Twenty-five more claims were kept out of the queue. They have the same blend of routine, hard and manual claims, and we did not look at them while we wrote the prompt, so they tell us whether a score carries over to claims we have not seen. Nothing is trained on either set. The queue plays the part of a development set and the held-out claims the part of a test set.

We run them once, with Luna at `medium`, and put the two scores together.

In [19]:
held_out_claims, held_out_labels = get_sample_data(tiny=TINY, held_out=True)
held_out_scored = score(run("Luna medium", held_out_claims), held_out_labels)

Luna medium: 25 claims in 18 seconds


In [20]:
comparison = pd.concat([summarize(scored[scored["model"] == "Luna medium"]), summarize(held_out_scored)])
comparison.index = ["Luna medium, the queue", "Luna medium, held out"]
comparison

,decisions_right,payouts_right,claims,cost,output_tokens,mean_seconds
"Luna medium, the queue",19,17,25,0.012705,5449,3.528
"Luna medium, held out",21,17,25,0.011621,5104,3.248


In [21]:
by_difficulty(held_out_scored)

difficulty,routine,hard,manual
model,,,
Luna medium,15,2,0


## Optional extras

Two more entries for the curious, both on the queue. Nothing later in the lab depends on them.

### Terra at high

GPT-5.6 Terra is Luna's larger sibling at ten times the price. At the `high` reasoning level it shows what a much bigger budget buys on this prompt.

In [22]:
LINEUP["Terra high"] = {"model": "openai/gpt-5.6-terra", "effort": "high"}

terra_scored = score(run("Terra high", claims), labels)
summarize(pd.concat([scored, terra_scored]))

Terra high: 25 claims in 16 seconds


,decisions_right,payouts_right,claims,cost,output_tokens,mean_seconds
model,,,,,,
Luna low,18,16,25,0.010831,4419,3.296
Qwen low,21,20,25,0.069774,33485,27.808
gpt-oss-20b medium,17,13,25,0.004893,22073,6.448
"gpt-oss-20b medium, local",16,12,25,0.000000,5151,22.344
Luna medium,19,17,25,0.012705,5449,3.528
Terra high,20,18,25,0.114662,4032,2.960


In [23]:
by_difficulty(terra_scored)

difficulty,routine,hard,manual
model,,,
Terra high,14,3,1


### Jev

Jev, from TypeSafe AI, is a different kind of model: it generates no text. We send it the facts as `state` with a typed question, and it answers with a probability for each option and a confidence. So it can choose between approve, decline and refer in a fraction of a second, and it cannot write a rationale or work out a payout. We score it on the decision alone, which compares with the `decisions_right` column above.

Jev sits on OpenRouter's Decisions endpoint, which the OpenAI client cannot reach, so we use OpenRouter's own Python client with the same key.

In [24]:
JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the documents it asks for are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def decide_with_jev(claim):
    """Ask Jev for the decision on one claim, with its confidence."""
    state = {"policy_wording": policy_wording, "policy": policies[claim["policy_id"]], "claim": claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2),
    }

In [25]:
with ThreadPoolExecutor(max_workers=5) as pool:
    jev = pd.DataFrame(list(pool.map(decide_with_jev, claims)))

jev["label"] = jev["claim_id"].map(lambda claim_id: labels[claim_id]["decision"])
jev["decision_right"] = jev["decision"] == jev["label"]

print(f"Jev: {jev['decision_right'].sum()} of {len(jev)} decisions right")
print(f"${jev['cost'].sum():.4f} for the set, {jev['seconds'].mean():.2f} seconds a claim")

Jev: 15 of 25 decisions right
$0.0033 for the set, 0.27 seconds a claim


Jev reports a confidence with each answer. Here are its decisions split at a confidence of 0.9, which points to one way of using a fast model: keep the answers it is sure of and pass the rest to a stronger model.

In [26]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["decision_right"].agg(["sum", "count"])

,sum,count
confident,,
False,12,21
True,3,4


## An agent with tools

So far every call has returned a structured output, and our code has done the rest: it found the policy record, put it in the prompt and read the decision from the reply. An agent does that work itself through tools. `create_agent` from LangChain runs the loop for us. The model asks for a tool, our code runs it and sends back the result, and the model carries on until it is done.

We give it two tools. `get_policy` looks up the customer's policy record, so the prompt now holds only the claim, which names its policy by id. `record_decision` records the outcome. Its arguments are typed by the function's signature, so the tool call takes the place of the structured output. Here it writes to a dictionary. In the UnderCover project it will write to the claims system.

In [27]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is two or three sentences that explain the decision and name the section of the policy wording it rests on.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent runs on Luna at `medium`, through LangChain's `ChatOpenAI` pointed at OpenRouter. Its system prompt keeps the instructions and the policy wording we have used all along, and the line about the order of the reply becomes a line about the tools. Once it has looked up the policy the agent has the same information as before, so the score compares with the `Luna medium` row.

In [28]:
luna = ChatOpenAI(
    model="openai/gpt-5.6-luna",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    reasoning_effort="medium",
    timeout=120,
    max_retries=5,
)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

One claim first, with every message after ours: the model asks for the policy, the tool returns it, the model records its decision, and it closes with a short message.

In [29]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_o9YLQOtu5O6SxNXee2J9klKj)
 Call ID: call_o9YLQOtu5O6SxNXee2J9klKj
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_qCKOujGSlPRQZHc9sA4a7tWO)
 Call ID: call_qCKOujGSlPRQZHc9sA4a7tWO
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420
    rationale: Approved under Section 3 (Trip Delay): Meridian Air 

Now the whole queue. Each claim takes three calls to the model: one that asks for the policy, one that records the decision and one to close. OpenRouter's cost for each is in the message's metadata. A claim with nothing recorded counts as a miss.

In [30]:
def decide_with_agent(claim):
    """Run the agent on one claim and read back what it recorded."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    usage = [message.response_metadata["token_usage"] for message in result["messages"] if message.type == "ai"]
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    return {
        "model": "Luna medium, agent",
        "claim_id": claim["claim_id"],
        **answer,
        "provider": "OpenAI",
        "cost": sum(call["cost"] for call in usage),
        "input_tokens": sum(call["prompt_tokens"] for call in usage),
        "output_tokens": sum(call["completion_tokens"] for call in usage),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
with ThreadPoolExecutor(max_workers=5) as pool:
    agent_results = list(pool.map(decide_with_agent, claims))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

25 of 25 claims have a recorded decision


In [31]:
agent_scored = score(agent_results, labels)
summarize(pd.concat([scored[scored["model"] == "Luna medium"], agent_scored]))

,decisions_right,payouts_right,claims,cost,output_tokens,mean_seconds
model,,,,,,
Luna medium,19,17,25,0.012705,5449,3.528
"Luna medium, agent",19,16,25,0.018656,8331,7.040


In [32]:
by_difficulty(agent_scored)

difficulty,routine,hard,manual
model,,,
"Luna medium, agent",15,1,0


In [33]:
show_misses(agent_scored, labels, "Luna medium, agent")

**CLM-1011, hard**: approve $1,110.00, where the handler said approve $1,150.00

- Model: The checked baggage was lost during the covered trip, and the carrier report and settlement letter satisfy the reporting/document requirements in Section 5; the carrier's $350 payment is deducted under Section 10. Applying Premium's $300 per-item limit (including the tuxedo as one suit and the camera with lens as one set), the eligible amount is $1,510; after the $350 carrier payment and $50 deductible, the payable amount is $1,110, within the $1,500 baggage maximum.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1014, hard**: approve $560.00, where the handler said approve $605.00

- Model: Approved under section 5, Baggage and Personal Effects: the carrier report confirms the checked baggage was lost during the covered trip, and the carrier’s $100 settlement is documented. Under section 10, eligible belongings total $760 after applying the $150 per-item limit to the three polo shirts, less the $100 carrier payment and the $100 Economy deductible, resulting in a $560 payout (within the $750 maximum).
- Handler: The carrier has declared the bag lost and its report is on file. The per-item limit on Economy is $150 and it applies to each item. The receipts show three shirts at $65, two swimsuits at $85 and two pairs of shorts at $60, so every item is under the limit and each line is paid in full, although two of the lines come to more than $150. The carrier's $100 comes off, then the $100 deductible, and the result is inside the $750 maximum.
- Tests: Payout arithmetic: the per-item limit applies to items, and a line that holds several items is not capped as one


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The theft occurred during the covered trip, was reported to the local police within 24 hours, and the Carabinieri report is attached, satisfying Section 5. Under Section 10, the jacket is limited to the Premium plan's $300 per-item limit and the daypack is eligible at $80; after the $50 baggage deductible, the payout is $330.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: Approved under Section 3 (Trip Delay): the carrier confirmed a crew problem causing a delay from 7:30 a.m. to 4:50 p.m., exceeding the Premium plan's six-hour waiting period. The attached receipts support $112 for accommodation, $58 for meals, and $40 for local transport; the $210 total is below the Premium $800 maximum, and no deductible applies.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: Refer because this is a theft claim under Section 5 (Baggage and Personal Effects), which requires the theft to be reported to local police within 24 hours and a copy of the police report. The submitted tour-guide letter and optician receipt do not include the required police report, so the claim cannot be decided until that documentation is provided or the reporting issue is reviewed.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: Approved under Section 5 (Baggage and Personal Effects): the theft occurred during the covered trip, was reported to local police the same day within 24 hours, and the police report is attached. The Electronics Plus add-on raises the electronics per-item limit to $1,200, so all listed items are eligible; under Section 10, $1,103 eligible loss less the Premium $50 deductible results in a $1,053 payout, within the $1,500 benefit maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: Refer under Section 3 (Trip Delay) and Section 11 because the carrier report confirms a qualifying severe-weather delay exceeding the First plan's 3-hour waiting period, but receipts are required for the claimed meal expenses and none were provided. The rebooking receipt is attached, but the claim cannot be finalized until the missing meal receipts are supplied.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: Approved under Section 3, Trip Delay: the carrier certificate confirms a mechanical delay from 08:15 to 17:32, exceeding the First plan's 3-hour waiting period. The attached meal receipts support $61 of reasonable extra costs, payable in full under Section 10 because no deductible applies and the amount is below the $1,500 Trip Delay maximum.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: Approved under Section 5, Baggage and Personal Effects: the smartphone was stolen during the covered trip and the police report confirms it was reported the same day, within 24 hours. Under Section 10, the eligible amount is limited to the Premium plan's $300 per-item limit; after the $50 deductible under Section 1, the payout is $250.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


## Where this leaves us

Luna gets nearly all of the routine claims right at either level. The smaller gpt-oss-20b drops a few of them and gets none of the hard claims, whether it runs hosted or on this machine. Qwen does better on the hard claims, costs several times as much and takes minutes where Luna takes seconds. The manual claims are nearly all wrong for every entry, because the rules they turn on are in the claims manual and the customer's history, and the prompt holds neither. Retrieval and tools take that on in Week 2, and the hard claims are what we work on when we improve the agent.

Luna scores about the same on the held-out claims as on the queue, and about the same when the agent works through its tools as it does with the structured output.

One run is a weak measurement. Run the line-up again and the scores move by a claim or two, so treat a small difference between two entries with suspicion until you have repeated it.